# Runs comparer

Cross-experiment comparison of sweep runs: metrics heatmap, covering-rank stats,
gold-rank table, late golds. Runs are loaded from every experiment's `runs/` folder;
select which to compare by label.

In [ ]:
from pathlib import Path
import pandas as pd

from minerva_eval import load_all_results, get_run_summary, metrics_heatmap, gold_ranks, covering_rank_stats, late_golds

ROOTS = sorted(Path("../experiments").glob("*/runs"))   # every experiment's runs
all_results = load_all_results(ROOTS)

collections = {r.collection for r in all_results}
print(f"{len(all_results)} rows from {len(collections)} collections: {collections}")

## Run summary

One row per run folder: collection, timestamp, sweep matrix, query count.

In [ ]:
run_summary = get_run_summary(ROOTS)
run_summary

In [ ]:
import ipywidgets as widgets

labels = sorted({r.label for r in all_results})
picker = widgets.SelectMultiple(options=labels, value=tuple(labels),
                                description="runs", rows=(len(labels) + 2), layout=widgets.Layout(width="700px"))
display(picker)

## Metrics view

Mean over queries, grouped by `(label, collection, hybrid_alpha, reranker_depth)`. Reproduces the
headline table from the sweep-findings spec; styled as a heatmap.

In [ ]:
chosen = [r for r in all_results if r.label in picker.value]
metrics_heatmap(chosen)

## Covering rank

Covering rank = smallest $k$ whose top-$k$ contains **all** the query's golds
(max over gold sources), so it is the depth a client has to read to get the
whole answer. One row per cell: `p90` = 4 reads as "90% of queries have every
gold within top 4". `p50` is 1 almost everywhere and says nothing; the cells
differ in the tail. `beyond` counts the queries whose gold never appeared
inside `top_k` (the sentinel `top_k + 1`), kept out of `max` so a sentinel
cannot pass for a rank.

The gold rank table below breaks these numbers down query by query.

In [ ]:
covering_rank_stats(chosen)

## Gold rank table

Value is the gold's worst source rank in that cell. Queries with multiple golds
collapse with `max` (the worst-placed gold). Rows are sorted hardest
first (largest worst-case rank on top), so trivially easy queries (rank 1 in
every cell) sink to the bottom. Green = low rank (easy), red = high rank (hard).

In [ ]:
ranks = pd.DataFrame(vars(g) for g in gold_ranks(chosen))

table = ranks.pivot_table(
    index="query_id", columns=["collection", "label", "alpha", "rerank_depth", "cascade_depth"],
    values="rank", 
    aggfunc="max"           # multi-gold -> all have been found?
    )

table = table.loc[table.max(axis=1).sort_values(ascending=False).index]

table.style.background_gradient(cmap="RdYlGn_r", axis=None)

## Late golds

One row per gold **source** beyond `threshold`, worst first, so a query that is
deep in the tables above can be read apart: is one source missing, or all of
them? A query with a single gold appears once; a two-gold query appears twice
when both are late. Queries whose golds are all within the threshold are absent.

In [ ]:
late_golds(chosen, threshold=10)